# PHASE 3 — SYNTHETIC DATA GENERATION

**Project:** BALANCEFLOW AI  
**Objective:** Generate 5,000 rows of urban mobility demand-supply data and 1,000 rows of user behavioral data to serve as the foundational training pipeline for our AI models.

## Dataset 1: Mobility Data
* **Target File:** `data/mobility_data.csv`
* **Target Size:** 5,000 rows
* **Features:** `district`, `hour`, `demand`, `supply`, `weather_score`, `event_score`, `historical_congestion`, `pool_acceptance`, `pickup_flexibility`, `fare_sensitivity`, `risk_score`

In [1]:
import pandas as pd
import numpy as np
import os

# Set random seed for reproducibility
np.random.seed(42)

num_mobility_rows = 5000

# Generate random districts (D1 to D12) and hours (0 to 23)
districts = np.random.choice([f"D{i}" for i in range(1, 13)], size=num_mobility_rows)
hours = np.random.randint(0, 24, size=num_mobility_rows)

# Simulate smart demand-supply logic (Peak hours: 7-9 AM, 5-7 PM -> High demand, low supply)
demand = []
supply = []
for h in hours:
    if h in [7, 8, 9, 17, 18, 19]:
        d = np.random.randint(130, 250)
        s = np.random.randint(50, 110)
    else:
        d = np.random.randint(20, 120)
        s = np.random.randint(30, 130)
    demand.append(d)
    supply.append(s)

demand = np.array(demand)
supply = np.array(supply)

# Generate scores bounded between 0.0 and 1.0
weather_score = np.round(np.random.uniform(0.0, 1.0, size=num_mobility_rows), 2)
event_score = np.round(np.random.uniform(0.0, 1.0, size=num_mobility_rows), 2)
historical_congestion = np.round(np.random.uniform(0.0, 1.0, size=num_mobility_rows), 2)
pool_acceptance_m = np.round(np.random.uniform(0.0, 1.0, size=num_mobility_rows), 2)
pickup_flexibility_m = np.round(np.random.uniform(0.0, 1.0, size=num_mobility_rows), 2)
fare_sensitivity_m = np.round(np.random.uniform(0.0, 1.0, size=num_mobility_rows), 2)

# Calculate risk_score based on demand-supply imbalance and historical bottlenecks
raw_risk = (demand / (supply + 1)) * 0.5 + historical_congestion * 0.5
risk_score = np.round(np.clip(raw_risk / raw_risk.max(), 0.0, 1.0), 2)

# Create DataFrame
df_mobility = pd.DataFrame({
    "district": districts,
    "hour": hours,
    "demand": demand,
    "supply": supply,
    "weather_score": weather_score,
    "event_score": event_score,
    "historical_congestion": historical_congestion,
    "pool_acceptance": pool_acceptance_m,
    "pickup_flexibility": pickup_flexibility_m,
    "fare_sensitivity": fare_sensitivity_m,
    "risk_score": risk_score
})

# Export and overwrite the placeholder file created in Phase 1
df_mobility.to_csv("../data/mobility_data.csv", index=False)
print(f"✅ Successfully generated {len(df_mobility)} rows in data/mobility_data.csv!")
df_mobility.head(3)  # Visual preview of the first 3 rows

✅ Successfully generated 5000 rows in data/mobility_data.csv!


,district,hour,demand,supply,weather_score,event_score,historical_congestion,pool_acceptance,pickup_flexibility,fare_sensitivity,risk_score
0,D7,14,34,121,0.14,0.29,0.92,0.14,0.81,0.03,0.22
1,D4,19,139,87,0.17,0.57,0.85,0.81,0.90,0.41,0.44
2,D11,9,176,106,0.05,0.42,0.80,0.93,0.79,0.55,0.45


## Dataset 2: User Behavior
* **Target File:** `data/users.csv`
* **Target Size:** 1,000 rows
* **Features:** `user_id`, `pool_acceptance`, `pickup_flexibility`, `fare_sensitivity`, `eco_score`, `adaptability_score`

### Adaptability Score Formula
$$adaptability\_score = (0.4 \times pool\_acceptance + 0.3 \times pickup\_flexibility + 0.3 \times fare\_sensitivity) \times 100$$

In [2]:
num_users = 1000
user_ids = [f"USER_{i:04d}" for i in range(1, num_users + 1)]

# Generate baseline user behavioral metrics (0.0 to 1.0)
pool_acceptance_u = np.round(np.random.uniform(0.0, 1.0, size=num_users), 2)
pickup_flexibility_u = np.round(np.random.uniform(0.0, 1.0, size=num_users), 2)
fare_sensitivity_u = np.round(np.random.uniform(0.0, 1.0, size=num_users), 2)
eco_score = np.round(np.random.uniform(0.0, 1.0, size=num_users), 2)

# Apply the exact formula from our project specifications
adaptability_score = np.round(
    (0.4 * pool_acceptance_u + 0.3 * pickup_flexibility_u + 0.3 * fare_sensitivity_u) * 100, 
    2
)

# Create DataFrame
df_users = pd.DataFrame({
    "user_id": user_ids,
    "pool_acceptance": pool_acceptance_u,
    "pickup_flexibility": pickup_flexibility_u,
    "fare_sensitivity": fare_sensitivity_u,
    "eco_score": eco_score,
    "adaptability_score": adaptability_score
})

# Export and overwrite the placeholder file created in Phase 1
df_users.to_csv("../data/users.csv", index=False)
print(f"✅ Successfully generated {len(df_users)} rows in data/users.csv!")
df_users.head(3)  # Visual preview of the first 3 rows

✅ Successfully generated 1000 rows in data/users.csv!


,user_id,pool_acceptance,pickup_flexibility,fare_sensitivity,eco_score,adaptability_score
0,USER_0001,0.56,0.79,0.41,0.09,58.4
1,USER_0002,0.89,0.29,0.37,0.01,55.4
2,USER_0003,0.57,0.08,0.46,0.37,39.0


# PHASE 4 — AI MODEL TRAINING

## 1. Risk Score Recalculation & Normalization
We apply the official project formula to compute the traffic risk profile across districts:
$$risk\_score = \left(\frac{demand}{supply} \times 40\right) + (weather\_score \times 20) + (event\_score \times 20) + (historical\_congestion \times 20)$$

The raw score is strictly normalized onto a scale of **0 to 100**.

## 2. Model Architecture
* **Algorithm:** Random Forest Regressor (`RandomForestRegressor`)
* **Features ($X$):** `demand`, `supply`, `weather_score`, `event_score`, `historical_congestion`
* **Target ($y$):** `risk_score`

In [4]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
import joblib

# 1. LOAD DATASET
print("⏳ Loading mobility data for AI training...")
df = pd.read_csv("../data/mobility_data.csv")

# 2. RECALCULATE RISK SCORE USING THE OFFICIAL FORMULA
# Adding a tiny epsilon (1e-5) to supply to prevent division by zero errors
raw_risk = ((df["demand"] / (df["supply"] + 1e-5)) * 40 + 
            df["weather_score"] * 20 + 
            df["event_score"] * 20 + 
            df["historical_congestion"] * 20)

# Normalize the raw risk score strictly from 0 to 100
min_val = raw_risk.min()
max_val = raw_risk.max()
df["risk_score"] = np.round(((raw_risk - min_val) / (max_val - min_val)) * 100, 2)

# Save the updated data back to csv
df.to_csv("../data/mobility_data.csv", index=False)
print("✅ Risk scores successfully updated and normalized (0-100)!")

# 3. DEFINE FEATURES AND TARGET
features = ["demand", "supply", "weather_score", "event_score", "historical_congestion"]
target = "risk_score"

X = df[features]
y = df[target]

# Split into Training and Testing sets (80% Train, 20% Test)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 4. TRAIN RANDOM FOREST REGRESSOR
print("⏳ Training RandomForestRegressor model...")
model = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)

# Evaluate model performance
y_pred = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)
print(f"📊 Model Evaluation -> RMSE: {rmse:.2f} | R² Score: {r2:.4f}")

# 5. SAVE THE TRAINED MODEL ARTIFACT
model_path = "../models/risk_model.pkl"
joblib.dump(model, model_path)
print(f"🚀 Model successfully saved to: {model_path}")

ImportError: DLL load failed while importing _weight_vector: An Application Control policy has blocked this file.